# Pipeline utilities

Shared code for the pipeline: table definitions, execution logging, quarantine,
MERGE (upsert) logic, the Silver -> Gold refresh and the parameterized `run_pipeline` function.
Loaded with `%run ./pipeline_utils` from `run_pipeline`, `pipeline_tests` and `gold_optimization`.

In [0]:
%run "./bronze_silver layer"

In [0]:
%run "./gold_layer"

In [0]:
import os
import glob
import uuid
from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, LongType,
    TimestampType, BooleanType, DateType
)

DEFAULT_SCHEMA = "world_bank"

# Default raw files for each load type (relative to the repo folder)
DEFAULT_SOURCES = {
    "full": "../data/raw/full",
    "incremental": "../data/raw/incremental",
}

# Gold tables, in the order they are refreshed (dimensions before the fact)
GOLD_KEYS = ["dim_country", "dim_indicator", "dim_year", "fact", "agg_trends", "agg_yoy", "agg_latest"]

def utc_now():
    return datetime.now(timezone.utc)


def get_tables(catalog="", schema=DEFAULT_SCHEMA):
    catalog = (catalog or "").strip() or spark.catalog.currentCatalog()
    schema = (schema or "").strip() or DEFAULT_SCHEMA
    prefix = f"{catalog}.{schema}"
    return {
        "schema": prefix,
        "bronze": f"{prefix}.bronze_indicator_values",
        "silver": f"{prefix}.silver_indicator_values",
        "revisions": f"{prefix}.silver_indicator_revisions",
        "quarantine": f"{prefix}.quarantine_records",
        "logs": f"{prefix}.pipeline_execution_logs",
        "dim_country": f"{prefix}.gold_dim_country",
        "dim_indicator": f"{prefix}.gold_dim_indicator",
        "dim_year": f"{prefix}.gold_dim_year",
        "fact": f"{prefix}.gold_fact_indicator_values",
        "agg_trends": f"{prefix}.gold_agg_indicator_trends",
        "agg_yoy": f"{prefix}.gold_agg_yoy_growth",
        "agg_latest": f"{prefix}.gold_agg_latest_values",
    }

In [0]:
def create_tables(tables):
    """Create the schema and all tables if they do not exist yet (safe to run every time)."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {tables['schema']}")
    is_new = {key: not spark.catalog.tableExists(tables[key]) for key in ["silver", "logs"]}

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['bronze']} (
            `indicator`       STRUCT<id: STRING, value: STRING> COMMENT 'Indicator code and name as sent by the API',
            `country`         STRUCT<id: STRING, value: STRING> COMMENT 'Country/region code and name as sent by the API',
            `countryiso3code` STRING COMMENT 'ISO3 code (empty for some aggregates)',
            `date`            STRING COMMENT 'Year, raw string',
            `value`           STRING COMMENT 'Indicator value, raw text (cast in Silver)',
            `unit`            STRING,
            `obs_status`      STRING,
            `decimal`         STRING COMMENT 'Decimal precision, raw text',
            `_rescued_data`   STRING COMMENT 'Fields that did not match the Bronze schema (schema drift)',
            `record_hash`     STRING NOT NULL COMMENT 'SHA-256 of the raw record content',
            `source_file`     STRING NOT NULL COMMENT 'Raw file the record came from',
            `batch_id`        STRING NOT NULL COMMENT 'Batch / load identifier (pipeline parameter)',
            `load_type`       STRING COMMENT 'full or incremental',
            `load_timestamp`  TIMESTAMP NOT NULL COMMENT 'When the record was ingested into Bronze'
        ) USING DELTA
        COMMENT 'Bronze: raw World Bank indicator records. Key: (batch_id, record_hash)'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['silver']} (
            `country_key`           STRING NOT NULL COMMENT 'Country id, or NAME_<name> when the API sends an empty id',
            `country_code`          STRING COMMENT 'World Bank country/region id',
            `country_iso3`          STRING COMMENT 'ISO3 code',
            `country_name`          STRING,
            `indicator_code`        STRING NOT NULL,
            `indicator_name`        STRING,
            `year`                  INT NOT NULL,
            `value`                 DOUBLE,
            `is_reported`           BOOLEAN COMMENT 'false when the API has no value for this country-year',
            `decimal_places`        INT,
            `source_batch_id`       STRING COMMENT 'Batch that last changed this row',
            `source_file`           STRING,
            `source_load_timestamp` TIMESTAMP COMMENT 'Bronze load time of the source record',
            `load_timestamp`        TIMESTAMP NOT NULL COMMENT 'When this row was last inserted/updated in Silver'
        ) USING DELTA
        COMMENT 'Silver: cleansed indicator values. Primary key: (country_key, indicator_code, year)'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['revisions']} (
            `country_key`     STRING NOT NULL,
            `indicator_code`  STRING NOT NULL,
            `year`            INT NOT NULL,
            `old_value`       DOUBLE,
            `new_value`       DOUBLE,
            `change_type`     STRING COMMENT 'newly_reported, value_revised or value_removed',
            `old_batch_id`    STRING,
            `new_batch_id`    STRING NOT NULL,
            `load_timestamp`  TIMESTAMP NOT NULL
        ) USING DELTA
        COMMENT 'History of value changes applied to Silver by incremental loads'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['quarantine']} (
            `layer`           STRING NOT NULL COMMENT 'raw_to_bronze or bronze_to_silver',
            `batch_id`        STRING,
            `source_file`     STRING,
            `reason`          STRING NOT NULL,
            `record_hash`     STRING,
            `raw_record`      STRING COMMENT 'The rejected record as JSON',
            `load_timestamp`  TIMESTAMP NOT NULL
        ) USING DELTA
        COMMENT 'Records rejected by validation (non-conforming / schema drift)'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['logs']} (
            `log_id`            STRING NOT NULL,
            `run_id`            STRING NOT NULL COMMENT 'Same for all steps of one pipeline run',
            `layer`             STRING NOT NULL COMMENT 'raw_to_bronze, bronze_to_silver or silver_to_gold',
            `load_type`         STRING,
            `batch_id`          STRING,
            `source`            STRING COMMENT 'File or table/parameter processed',
            `start_time`        TIMESTAMP,
            `end_time`          TIMESTAMP,
            `duration_seconds`  DOUBLE,
            `status`            STRING COMMENT 'SUCCESS or FAILED',
            `rows_read`         BIGINT,
            `rows_inserted`     BIGINT,
            `rows_updated`      BIGINT,
            `rows_quarantined`  BIGINT,
            `rows_aggregated`   BIGINT COMMENT 'Rows produced by a Silver -> Gold step',
            `message`           STRING,
            `load_timestamp`    TIMESTAMP NOT NULL
        ) USING DELTA
        COMMENT 'One row per file/table processed by every pipeline run'
    """)

    # Informational primary keys (supported on Unity Catalog only), added when the table is created
    for key, name, cols in [
        ("silver", "pk_silver_indicator_values", "country_key, indicator_code, year"),
        ("logs", "pk_pipeline_execution_logs", "log_id"),
    ]:
        if not is_new[key]:
            continue
        try:
            spark.sql(f"ALTER TABLE {tables[key]} ADD CONSTRAINT {name} PRIMARY KEY ({cols})")
        except Exception:
            pass  # not a Unity Catalog table

    # rows_aggregated was added with the Gold layer, log tables created before that get it here
    if not is_new["logs"] and "rows_aggregated" not in spark.table(tables["logs"]).columns:
        spark.sql(f"""
            ALTER TABLE {tables['logs']} ADD COLUMNS (
                `rows_aggregated` BIGINT COMMENT 'Rows produced by a Silver -> Gold step' AFTER `rows_quarantined`
            )
        """)

    create_gold_tables(tables)


def drop_all_tables(tables):
    """Remove all pipeline tables. Only for a clean demo/test run."""
    # fact and aggregate tables first, they hold the foreign keys to the dimensions
    for key in ["fact", "agg_trends", "agg_yoy", "agg_latest", "dim_country", "dim_indicator", "dim_year",
                "bronze", "silver", "revisions", "quarantine", "logs"]:
        spark.sql(f"DROP TABLE IF EXISTS {tables[key]}")

In [0]:
def create_gold_tables(tables):
    """Gold star schema. Called by create_tables, safe to run every time.

    Dimensions get an identity column as surrogate key. The fact and the aggregate tables
    use liquid clustering on the keys the dashboard filters on. The dimensions are a few
    hundred rows at most, so they are not clustered (Spark broadcasts them in joins).
    """
    is_new = {key: not spark.catalog.tableExists(tables[key]) for key in GOLD_KEYS}

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['dim_country']} (
            `country_sk`          BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key',
            `country_code`        STRING NOT NULL COMMENT 'World Bank id (business key). ISO3 ids like ARG are mapped to AR',
            `country_name`        STRING,
            `iso3_code`           STRING,
            `entity_type`         STRING COMMENT 'Country, Aggregate or Unclassified',
            `region`              STRING COMMENT 'World Bank region (countries only)',
            `income_level`        STRING COMMENT 'World Bank income group (countries only)',
            `lending_type`        STRING COMMENT 'IBRD / IDA / Blend (countries only)',
            `capital_city`        STRING,
            `latitude`            DOUBLE,
            `longitude`           DOUBLE,
            `source_country_keys` STRING COMMENT 'Silver country_key value(s) mapped to this row',
            `load_timestamp`      TIMESTAMP NOT NULL COMMENT 'When this row was last inserted/updated'
        ) USING DELTA
        COMMENT 'Gold: country and aggregate dimension (SCD type 1). Primary key: country_sk'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['dim_indicator']} (
            `indicator_sk`    BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key',
            `indicator_code`  STRING NOT NULL COMMENT 'World Bank indicator code (business key)',
            `indicator_name`  STRING,
            `category`        STRING COMMENT 'Theme, from the code prefix',
            `source_dataset`  STRING COMMENT 'World Bank database the indicator comes from',
            `unit`            STRING COMMENT 'Unit, from the indicator name',
            `is_additive`     BOOLEAN COMMENT 'true when values can be summed across countries',
            `load_timestamp`  TIMESTAMP NOT NULL COMMENT 'When this row was last inserted/updated'
        ) USING DELTA
        COMMENT 'Gold: indicator dimension (SCD type 1). Primary key: indicator_sk'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['dim_year']} (
            `year_key`          INT NOT NULL COMMENT 'The year itself, e.g. 2023',
            `decade`            INT COMMENT 'e.g. 2020',
            `decade_label`      STRING COMMENT 'e.g. 2020s',
            `five_year_period`  STRING COMMENT 'e.g. 2020-2024',
            `load_timestamp`    TIMESTAMP NOT NULL
        ) USING DELTA
        COMMENT 'Gold: time dimension at year grain. Primary key: year_key'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['fact']} (
            `country_sk`             BIGINT NOT NULL COMMENT 'FK to gold_dim_country',
            `indicator_sk`           BIGINT NOT NULL COMMENT 'FK to gold_dim_indicator',
            `year_key`               INT NOT NULL COMMENT 'FK to gold_dim_year',
            `value`                  DOUBLE COMMENT 'Indicator value (NULL when not reported)',
            `is_reported`            BOOLEAN,
            `decimal_places`         INT,
            `source_batch_id`        STRING COMMENT 'Batch that last changed the Silver row',
            `silver_load_timestamp`  TIMESTAMP NOT NULL COMMENT 'load_timestamp of the Silver row (incremental watermark)',
            `load_timestamp`         TIMESTAMP NOT NULL COMMENT 'When this row was last inserted/updated in Gold'
        ) USING DELTA
        CLUSTER BY (`indicator_sk`, `year_key`)
        COMMENT 'Gold: one row per country, indicator and year. Primary key: (country_sk, indicator_sk, year_key)'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['agg_trends']} (
            `group_type`           STRING NOT NULL COMMENT 'Region, Income level or World',
            `group_name`           STRING NOT NULL COMMENT 'e.g. South Asia, Low income, World',
            `indicator_sk`         BIGINT NOT NULL COMMENT 'FK to gold_dim_indicator',
            `year_key`             INT NOT NULL COMMENT 'FK to gold_dim_year',
            `country_count`        BIGINT COMMENT 'Countries in the group',
            `reporting_countries`  BIGINT COMMENT 'Countries with a value',
            `coverage_pct`         DOUBLE COMMENT 'reporting_countries / country_count * 100',
            `avg_value`            DOUBLE COMMENT 'Simple average across countries',
            `median_value`         DOUBLE COMMENT 'Approximate median',
            `min_value`            DOUBLE,
            `max_value`            DOUBLE,
            `sum_value`            DOUBLE COMMENT 'Total, only for additive indicators',
            `load_timestamp`       TIMESTAMP NOT NULL
        ) USING DELTA
        CLUSTER BY (`indicator_sk`, `group_type`)
        COMMENT 'Gold: indicator values aggregated per region, income level and world'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['agg_yoy']} (
            `country_sk`             BIGINT NOT NULL COMMENT 'FK to gold_dim_country',
            `indicator_sk`           BIGINT NOT NULL COMMENT 'FK to gold_dim_indicator',
            `year_key`               INT NOT NULL COMMENT 'FK to gold_dim_year',
            `value`                  DOUBLE,
            `prev_year_key`          INT COMMENT 'Previous year with a reported value',
            `prev_value`             DOUBLE,
            `years_gap`              INT COMMENT 'year_key - prev_year_key',
            `abs_change`             DOUBLE COMMENT 'value - prev_value',
            `pct_change`             DOUBLE COMMENT 'Change in % of prev_value',
            `annualized_growth_pct`  DOUBLE COMMENT 'Average growth per year over the gap (positive values only)',
            `load_timestamp`         TIMESTAMP NOT NULL
        ) USING DELTA
        CLUSTER BY (`indicator_sk`, `country_sk`)
        COMMENT 'Gold: year-over-year change of every reported value'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tables['agg_latest']} (
            `country_sk`       BIGINT NOT NULL COMMENT 'FK to gold_dim_country',
            `indicator_sk`     BIGINT NOT NULL COMMENT 'FK to gold_dim_indicator',
            `latest_year_key`  INT NOT NULL COMMENT 'FK to gold_dim_year: last year with a reported value',
            `latest_value`     DOUBLE COMMENT 'Value in latest_year_key',
            `first_year_key`   INT COMMENT 'First year with a reported value',
            `reported_years`   BIGINT COMMENT 'Number of years with a value',
            `load_timestamp`   TIMESTAMP NOT NULL
        ) USING DELTA
        CLUSTER BY (`indicator_sk`)
        COMMENT 'Gold: latest available value per country and indicator'
    """)

    # Informational primary / foreign keys (Unity Catalog only), added when the tables are
    # created. They are not enforced, but Catalog Explorer and BI tools use them to show
    # the relationships of the star schema.
    dim_country, dim_indicator, dim_year = tables["dim_country"], tables["dim_indicator"], tables["dim_year"]
    constraints = [
        ("dim_country", "pk_gold_dim_country", "PRIMARY KEY (country_sk)"),
        ("dim_indicator", "pk_gold_dim_indicator", "PRIMARY KEY (indicator_sk)"),
        ("dim_year", "pk_gold_dim_year", "PRIMARY KEY (year_key)"),
        ("fact", "pk_gold_fact_indicator_values", "PRIMARY KEY (country_sk, indicator_sk, year_key)"),
        ("fact", "fk_fact_country", f"FOREIGN KEY (country_sk) REFERENCES {dim_country}"),
        ("fact", "fk_fact_indicator", f"FOREIGN KEY (indicator_sk) REFERENCES {dim_indicator}"),
        ("fact", "fk_fact_year", f"FOREIGN KEY (year_key) REFERENCES {dim_year}"),
        ("agg_trends", "pk_gold_agg_indicator_trends", "PRIMARY KEY (group_type, group_name, indicator_sk, year_key)"),
        ("agg_trends", "fk_trends_indicator", f"FOREIGN KEY (indicator_sk) REFERENCES {dim_indicator}"),
        ("agg_trends", "fk_trends_year", f"FOREIGN KEY (year_key) REFERENCES {dim_year}"),
        ("agg_yoy", "pk_gold_agg_yoy_growth", "PRIMARY KEY (country_sk, indicator_sk, year_key)"),
        ("agg_yoy", "fk_yoy_country", f"FOREIGN KEY (country_sk) REFERENCES {dim_country}"),
        ("agg_yoy", "fk_yoy_indicator", f"FOREIGN KEY (indicator_sk) REFERENCES {dim_indicator}"),
        ("agg_yoy", "fk_yoy_year", f"FOREIGN KEY (year_key) REFERENCES {dim_year}"),
        ("agg_latest", "pk_gold_agg_latest_values", "PRIMARY KEY (country_sk, indicator_sk)"),
        ("agg_latest", "fk_latest_country", f"FOREIGN KEY (country_sk) REFERENCES {dim_country}"),
        ("agg_latest", "fk_latest_indicator", f"FOREIGN KEY (indicator_sk) REFERENCES {dim_indicator}"),
        ("agg_latest", "fk_latest_year", f"FOREIGN KEY (latest_year_key) REFERENCES {dim_year}"),
    ]
    for key, name, definition in constraints:
        if not is_new[key]:
            continue
        try:
            spark.sql(f"ALTER TABLE {tables[key]} ADD CONSTRAINT {name} {definition}")
        except Exception:
            pass  # not a Unity Catalog table

In [0]:
LOG_SCHEMA = StructType([
    StructField("log_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("rows_aggregated", LongType(), True),
    StructField("message", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])


def write_log(tables, entry):
    row = tuple(entry.get(f.name) for f in LOG_SCHEMA.fields)
    spark.createDataFrame([row], LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(tables["logs"])


def run_step(tables, run_id, layer, load_type, batch_id, source, step_fn):
    """Run one pipeline step and always write a log entry, whether it succeeds or fails.

    A failure is recorded and returned, it does not stop the other files of the batch.
    """
    entry = {
        "log_id": str(uuid.uuid4()),
        "run_id": run_id,
        "layer": layer,
        "load_type": load_type,
        "batch_id": batch_id,
        "source": source,
        "start_time": utc_now(),
    }
    try:
        metrics = step_fn() or {}
        entry.update(metrics)
        entry["status"] = "SUCCESS"
    except Exception as e:
        entry["status"] = "FAILED"
        entry["message"] = f"{type(e).__name__}: {e}"[:4000]
    finally:
        end = utc_now()
        entry["end_time"] = end
        entry["duration_seconds"] = round((end - entry["start_time"]).total_seconds(), 3)
        entry["load_timestamp"] = end
        write_log(tables, entry)
    print(f"[{entry['status']}] {layer} | {source} | read={entry.get('rows_read')} "
          f"inserted={entry.get('rows_inserted')} updated={entry.get('rows_updated')} "
          f"quarantined={entry.get('rows_quarantined')} aggregated={entry.get('rows_aggregated')} "
          f"| {entry.get('message') or ''}")
    return entry

In [0]:
def merge_into(merge_sql, target_table):
    """Run a MERGE statement and return (rows_inserted, rows_updated)."""
    result = spark.sql(merge_sql).collect()
    if result and "num_inserted_rows" in result[0].asDict():
        row = result[0].asDict()
        return int(row.get("num_inserted_rows") or 0), int(row.get("num_updated_rows") or 0)
    # Fallback: read the metrics of the last commit from the Delta history
    metrics = spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").collect()[0]["operationMetrics"]
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))


def quarantine_records(tables, df, layer, reason_col, raw_cols, load_ts):
    """Write rejected records to the quarantine table (MERGE, so a re-run adds no duplicates)."""
    q = (
        df.select(
            F.lit(layer).alias("layer"),
            F.col("batch_id"),
            F.col("source_file"),
            F.col(reason_col).alias("reason"),
            F.col("record_hash"),
            F.to_json(F.struct(*raw_cols)).alias("raw_record"),
            F.lit(load_ts).cast(TimestampType()).alias("load_timestamp"),
        )
        .dropDuplicates(["layer", "batch_id", "record_hash", "reason"])
    )
    q.createOrReplaceTempView("_quarantine_src")
    inserted, _ = merge_into(f"""
        MERGE INTO {tables['quarantine']} t
        USING _quarantine_src s
        ON  t.layer = s.layer AND t.batch_id <=> s.batch_id
        AND t.record_hash <=> s.record_hash AND t.reason = s.reason
        WHEN NOT MATCHED THEN INSERT *
    """, tables["quarantine"])
    return inserted

In [0]:
def load_file_to_bronze(tables, path, batch_id, load_type):
    """Raw file -> Bronze for one file."""
    load_ts = utc_now()
    file_name = os.path.basename(path)

    staged = (
        add_bronze_metadata(read_raw_file(path), file_name, batch_id, load_type, load_ts)
        .withColumn("_reject_reason", bronze_reject_reason())
    )

    stats = staged.agg(
        F.count(F.lit(1)).alias("rows_read"),
        F.count("_reject_reason").alias("rejected"),
        F.count("_rescued_data").alias("drift_rows"),
    ).collect()[0]

    if stats["rows_read"] == 0:
        raise ValueError(f"No records could be read from {path}")

    if stats["rejected"] > 0:
        quarantine_records(
            tables, staged.filter(F.col("_reject_reason").isNotNull()),
            "raw_to_bronze", "_reject_reason", RAW_FIELDS + ["_rescued_data"], load_ts
        )

    good = (
        staged.filter(F.col("_reject_reason").isNull())
        .drop("_reject_reason")
        .dropDuplicates(["batch_id", "record_hash"])
    )
    good.createOrReplaceTempView("_bronze_src")

    # Insert-only MERGE: the same raw record in the same batch is never loaded twice.
    # WITH SCHEMA EVOLUTION lets the table pick up new columns if bronze_schema is extended.
    inserted, _ = merge_into(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {tables['bronze']} t
        USING _bronze_src s
        ON t.batch_id = s.batch_id AND t.record_hash = s.record_hash
        WHEN NOT MATCHED THEN INSERT *
    """, tables["bronze"])

    notes = []
    if stats["drift_rows"] > 0:
        notes.append(f"schema drift: {stats['drift_rows']} rows had unexpected/mismatched fields (kept in _rescued_data)")
    if stats["rejected"] > 0:
        notes.append(f"{stats['rejected']} rows quarantined")

    return {
        "rows_read": stats["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": 0,
        "rows_quarantined": stats["rejected"],
        "message": "; ".join(notes) or None,
    }

In [0]:
def load_bronze_to_silver(tables, batch_id):
    """Bronze -> Silver for one batch (or batch_id='ALL' to rebuild from every batch)."""
    load_ts = utc_now()

    bronze = spark.table(tables["bronze"])
    if batch_id.upper() != "ALL":
        bronze = bronze.filter(F.col("batch_id") == batch_id)

    staged = to_silver(bronze, load_ts).withColumn("_reject_reason", silver_reject_reason())

    stats = staged.agg(
        F.count(F.lit(1)).alias("rows_read"),
        F.count("_reject_reason").alias("rejected"),
    ).collect()[0]

    if stats["rows_read"] == 0:
        raise ValueError(f"No Bronze records found for batch_id = {batch_id}")

    if stats["rejected"] > 0:
        quarantine_records(
            tables,
            staged.filter(F.col("_reject_reason").isNotNull()).withColumnRenamed("source_batch_id", "batch_id"),
            "bronze_to_silver", "_reject_reason",
            ["country_code", "country_name", "indicator_code", "year_raw", "value_raw", "decimal_raw"], load_ts
        )

    source = latest_per_key(staged.filter(F.col("_reject_reason").isNull())).select(*SILVER_COLUMNS)
    source.createOrReplaceTempView("_silver_src")

    # A row is only changed when the incoming record is at least as new as the one
    # already in Silver. This way re-running an old batch (backfill) cannot overwrite
    # newer values, and re-running the same batch changes nothing.
    change_condition = """
        s.source_load_timestamp >= t.source_load_timestamp AND (
            NOT (t.value <=> s.value)
            OR NOT (t.country_name <=> s.country_name)
            OR NOT (t.country_iso3 <=> s.country_iso3)
            OR NOT (t.indicator_name <=> s.indicator_name)
            OR NOT (t.decimal_places <=> s.decimal_places)
        )
    """

    # 1) Record value revisions before Silver is updated
    revisions, _ = merge_into(f"""
        MERGE INTO {tables['revisions']} r
        USING (
            SELECT s.country_key, s.indicator_code, s.year,
                   t.value AS old_value, s.value AS new_value,
                   CASE WHEN t.value IS NULL THEN 'newly_reported'
                        WHEN s.value IS NULL THEN 'value_removed'
                        ELSE 'value_revised' END AS change_type,
                   t.source_batch_id AS old_batch_id,
                   s.source_batch_id AS new_batch_id,
                   s.load_timestamp
            FROM _silver_src s
            JOIN {tables['silver']} t
              ON t.country_key = s.country_key AND t.indicator_code = s.indicator_code AND t.year = s.year
            WHERE s.source_load_timestamp >= t.source_load_timestamp
              AND NOT (t.value <=> s.value)
        ) c
        ON  r.country_key = c.country_key AND r.indicator_code = c.indicator_code
        AND r.year = c.year AND r.new_batch_id = c.new_batch_id
        WHEN NOT MATCHED THEN INSERT *
    """, tables["revisions"])

    # 2) Upsert into Silver
    inserted, updated = merge_into(f"""
        MERGE INTO {tables['silver']} t
        USING _silver_src s
        ON t.country_key = s.country_key AND t.indicator_code = s.indicator_code AND t.year = s.year
        WHEN MATCHED AND {change_condition} THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """, tables["silver"])

    notes = [f"{revisions} value revisions recorded"]
    if stats["rejected"] > 0:
        notes.append(f"{stats['rejected']} rows quarantined")

    return {
        "rows_read": stats["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": updated,
        "rows_quarantined": stats["rejected"],
        "message": "; ".join(notes),
    }

In [0]:
def merge_dimension(target, source, key, attrs):
    """SCD type 1 MERGE: insert new members, overwrite the attributes that changed.

    The surrogate key is an identity column filled by the table, so it is never in the
    INSERT list and a member keeps the same key on every load.
    """
    source.select(key, *attrs, "load_timestamp").createOrReplaceTempView("_dim_src")
    cols = [key] + attrs + ["load_timestamp"]
    changed = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in attrs)
    updates = ", ".join(f"t.{c} = s.{c}" for c in attrs + ["load_timestamp"])
    insert_cols = ", ".join(cols)
    insert_values = ", ".join(f"s.{c}" for c in cols)
    return merge_into(f"""
        MERGE INTO {target} t
        USING _dim_src s
        ON t.{key} = s.{key}
        WHEN MATCHED AND ({changed}) THEN UPDATE SET {updates}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_values})
    """, target)


def load_gold_dim_country(tables, data_dir):
    load_ts = utc_now()
    reference, aliases = read_country_reference(data_dir)
    source = build_dim_country(spark.table(tables["silver"]), reference, aliases, load_ts)

    stats = source.agg(
        F.sum("silver_rows").alias("rows_read"),
        F.count(F.lit(1)).alias("members"),
        F.count(F.when(F.col("entity_type") == "Unclassified", 1)).alias("unclassified"),
    ).collect()[0]
    if not stats["rows_read"]:
        raise ValueError(f"{tables['silver']} is empty, run the Bronze -> Silver step first")

    inserted, updated = merge_dimension(tables["dim_country"], source, "country_code", DIM_COUNTRY_ATTRS)
    notes = [f"{stats['members']} countries / aggregates"]
    if stats["unclassified"]:
        notes.append(f"{stats['unclassified']} not in {COUNTRY_METADATA_FILE} (entity_type = Unclassified)")
    return {
        "rows_read": stats["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": updated,
        "rows_aggregated": stats["members"],
        "message": "; ".join(notes),
    }


def load_gold_dim_indicator(tables):
    load_ts = utc_now()
    source = build_dim_indicator(spark.table(tables["silver"]), load_ts)
    stats = source.agg(F.sum("silver_rows").alias("rows_read"), F.count(F.lit(1)).alias("members")).collect()[0]
    inserted, updated = merge_dimension(tables["dim_indicator"], source, "indicator_code", DIM_INDICATOR_ATTRS)
    return {
        "rows_read": stats["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": updated,
        "rows_aggregated": stats["members"],
        "message": f"{stats['members']} indicators",
    }


def load_gold_dim_year(tables):
    load_ts = utc_now()
    bounds = spark.table(tables["silver"]).agg(
        F.count(F.lit(1)).alias("rows_read"), F.min("year").alias("first_year"), F.max("year").alias("last_year")
    ).collect()[0]
    source = build_dim_year(bounds["first_year"], bounds["last_year"], load_ts)
    inserted, updated = merge_dimension(tables["dim_year"], source, "year_key", DIM_YEAR_ATTRS)
    years = bounds["last_year"] - bounds["first_year"] + 1
    return {
        "rows_read": bounds["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": updated,
        "rows_aggregated": years,
        "message": f"{years} years ({bounds['first_year']}-{bounds['last_year']})",
    }


def load_gold_fact(tables, data_dir):
    """Silver -> fact, incremental.

    Only the Silver rows inserted or updated after the last Gold refresh are read. The
    watermark is the newest silver_load_timestamp already in the fact, so a re-run with
    no Silver changes reads 0 rows, and an empty fact means a first (full) load.
    """
    load_ts = utc_now()
    watermark = spark.sql(
        f"SELECT CAST(MAX(silver_load_timestamp) AS STRING) AS w FROM {tables['fact']}"
    ).collect()[0]["w"]

    silver = spark.table(tables["silver"])
    if watermark is not None:
        silver = silver.filter(F.col("load_timestamp") > F.lit(watermark).cast(TimestampType()))

    _, aliases = read_country_reference(data_dir)
    mapped = map_fact_keys(silver, aliases, spark.table(tables["dim_country"]), spark.table(tables["dim_indicator"]))
    stats = mapped.agg(
        F.count(F.lit(1)).alias("rows_read"),
        F.count(F.when(F.col("country_sk").isNull(), 1)).alias("no_country"),
        F.count(F.when(F.col("indicator_sk").isNull(), 1)).alias("no_indicator"),
    ).collect()[0]

    scope = f"Silver rows changed after {watermark}" if watermark else "first load, all Silver rows"
    if stats["rows_read"] == 0:
        return {"rows_read": 0, "rows_inserted": 0, "rows_updated": 0, "rows_aggregated": 0,
                "message": "no Silver changes since the last Gold refresh"}
    if stats["no_country"] or stats["no_indicator"]:
        raise ValueError(f"{stats['no_country']} rows without a country and {stats['no_indicator']} "
                         f"rows without an indicator in the Gold dimensions")

    source = fact_source(mapped, load_ts)
    source.createOrReplaceTempView("_fact_src")
    rows = source.count()

    # Silver only changes a row when its content changed, so a newer Silver row always
    # replaces the Gold row. Re-running an old batch does not touch Silver, and so not Gold.
    inserted, updated = merge_into(f"""
        MERGE INTO {tables['fact']} t
        USING _fact_src s
        ON  t.country_sk = s.country_sk AND t.indicator_sk = s.indicator_sk AND t.year_key = s.year_key
        WHEN MATCHED AND s.silver_load_timestamp > t.silver_load_timestamp THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """, tables["fact"])

    return {
        "rows_read": stats["rows_read"],
        "rows_inserted": inserted,
        "rows_updated": updated,
        "rows_aggregated": rows,
        "message": f"incremental MERGE ({scope})",
    }


def latest_load(table):
    return spark.sql(f"SELECT MAX(load_timestamp) AS ts FROM {table}").collect()[0]["ts"]


def rebuild_gold_aggregate(tables, key, inputs, build):
    """Rebuild an aggregate table with INSERT OVERWRITE (one atomic Delta commit).

    A changed value also changes averages and growth rates of other rows, so these tables
    are recomputed as a whole. That only happens when one of the input tables changed after
    the last rebuild, so re-running the pipeline on the same data leaves them untouched.
    """
    built = latest_load(tables[key])
    changes = [ts for ts in (latest_load(tables[k]) for k in inputs) if ts is not None]
    if built is not None and not any(ts > built for ts in changes):
        return {"rows_read": 0, "rows_inserted": 0, "rows_updated": 0, "rows_aggregated": 0,
                "message": "inputs unchanged since the last rebuild, skipped"}

    load_ts = utc_now()
    source = build(load_ts)
    source.createOrReplaceTempView("_agg_src")
    rows = source.count()
    spark.sql(f"INSERT OVERWRITE TABLE {tables[key]} SELECT * FROM _agg_src")
    return {
        "rows_read": spark.table(tables["fact"]).count(),
        "rows_inserted": rows,
        "rows_updated": 0,
        "rows_aggregated": rows,
        "message": "rebuilt with INSERT OVERWRITE",
    }


def optimize_gold(tables, keys):
    """OPTIMIZE the Gold tables that changed. With liquid clustering only the files that are
    not clustered yet are rewritten, so this stays cheap on incremental runs."""
    for key in keys:
        spark.sql(f"OPTIMIZE {tables[key]}")
    return {"message": "OPTIMIZE (liquid clustering): " + ", ".join(tables[k] for k in keys)}


def refresh_gold(tables, run_id, load_type, batch_id, data_dir=None):
    """Silver -> Gold: dimensions first, then the fact, then the aggregate tables.

    Every table is its own logged step (layer = silver_to_gold). If a step fails, the steps
    after it are skipped, so the fact is never loaded against dimensions that did not refresh.
    """
    data_dir = data_dir or os.getcwd()
    fact_inputs = ["fact", "dim_country", "dim_indicator"]
    steps = {
        "dim_country": (tables["silver"], lambda: load_gold_dim_country(tables, data_dir)),
        "dim_indicator": (tables["silver"], lambda: load_gold_dim_indicator(tables)),
        "dim_year": (tables["silver"], lambda: load_gold_dim_year(tables)),
        "fact": (tables["silver"], lambda: load_gold_fact(tables, data_dir)),
        "agg_trends": (tables["fact"], lambda: rebuild_gold_aggregate(
            tables, "agg_trends", fact_inputs, lambda ts: build_agg_trends(
                spark.table(tables["fact"]), spark.table(tables["dim_country"]),
                spark.table(tables["dim_indicator"]), ts))),
        "agg_yoy": (tables["fact"], lambda: rebuild_gold_aggregate(
            tables, "agg_yoy", ["fact"], lambda ts: build_agg_yoy(spark.table(tables["fact"]), ts))),
        "agg_latest": (tables["fact"], lambda: rebuild_gold_aggregate(
            tables, "agg_latest", ["fact"], lambda ts: build_agg_latest(spark.table(tables["fact"]), ts))),
    }

    results, changed = [], []
    for key in GOLD_KEYS:
        source, step_fn = steps[key]
        entry = run_step(tables, run_id, "silver_to_gold", load_type, batch_id,
                         f"{source} -> {tables[key]}", step_fn)
        results.append(entry)
        if entry["status"] != "SUCCESS":
            return results
        if key in ("fact", "agg_trends", "agg_yoy", "agg_latest") and (entry.get("rows_inserted") or entry.get("rows_updated")):
            changed.append(key)

    if changed:
        results.append(run_step(tables, run_id, "silver_to_gold", load_type, batch_id,
                                "OPTIMIZE gold tables", lambda: optimize_gold(tables, changed)))
    return results

In [0]:
def resolve_source_files(source_path, load_type, data_dir=None):
    """Turn the source_path parameter into a list of files.

    source_path can be empty (default files for the load type), a file, a folder
    (all .json files inside), a wildcard pattern, or several of these separated by commas.
    Relative paths are resolved against data_dir (default: the repo folder).
    """
    data_dir = data_dir or os.getcwd()
    entries = [s.strip() for s in (source_path or "").split(",") if s.strip()] or [DEFAULT_SOURCES[load_type]]
    files = []
    for entry in entries:
        if entry.startswith("dbfs:") or "://" in entry:
            files.append(entry)  # cloud/DBFS path, passed to Spark as it is
            continue
        path = entry if os.path.isabs(entry) else os.path.join(data_dir, entry)
        if os.path.isdir(path):
            files.extend(sorted(glob.glob(os.path.join(path, "*.json"))))
        else:
            files.extend(sorted(glob.glob(path)))
    return list(dict.fromkeys(files))


def default_batch_id(load_type, run_ts):
    # full load is one baseline; incremental pulls get one batch per day
    return "full_load" if load_type == "full" else f"incremental_{run_ts:%Y%m%d}"


def run_pipeline(load_type="incremental", source_path="", batch_id="", layer="all",
                 catalog="", schema=DEFAULT_SCHEMA, data_dir=None):
    """Run the pipeline with parameters.

    load_type   : 'full' or 'incremental'
    source_path : raw file(s) / folder to load. Empty = default files for the load type
    batch_id    : batch identifier. Empty = 'full_load' or 'incremental_<YYYYMMDD>'.
                  With layer='silver' it selects which Bronze batch to process
                  ('ALL' = every batch)
    layer       : 'all'    raw -> bronze -> silver -> gold
                  'bronze' raw -> bronze only
                  'silver' bronze -> silver -> gold
                  'gold'   silver -> gold only
    catalog / schema : where the tables live
    """
    load_type = (load_type or "").strip().lower()
    layer = (layer or "all").strip().lower()
    if load_type not in DEFAULT_SOURCES:
        raise ValueError("load_type must be 'full' or 'incremental'")
    if layer not in ("all", "bronze", "silver", "gold"):
        raise ValueError("layer must be 'all', 'bronze', 'silver' or 'gold'")

    tables = get_tables(catalog, schema)
    create_tables(tables)

    run_id = str(uuid.uuid4())
    batch_id = (batch_id or "").strip() or default_batch_id(load_type, utc_now())
    if layer in ("all", "bronze") and batch_id.upper() == "ALL":
        raise ValueError("batch_id 'ALL' can only be used with layer = 'silver' or 'gold'")

    print(f"run_id={run_id} | load_type={load_type} | batch_id={batch_id} | layer={layer} | tables={tables['schema']}")
    results = []

    if layer in ("all", "bronze"):
        files = resolve_source_files(source_path, load_type, data_dir)
        if not files:
            def no_files():
                raise FileNotFoundError(f"No source files found for source_path='{source_path or DEFAULT_SOURCES[load_type]}'")
            results.append(run_step(tables, run_id, "raw_to_bronze", load_type, batch_id,
                                    source_path or DEFAULT_SOURCES[load_type], no_files))
        for path in files:
            results.append(run_step(
                tables, run_id, "raw_to_bronze", load_type, batch_id, path,
                lambda p=path: load_file_to_bronze(tables, p, batch_id, load_type)
            ))

    if layer in ("all", "silver"):
        loaded = any(r["status"] == "SUCCESS" for r in results) or layer == "silver"
        if loaded:
            results.append(run_step(
                tables, run_id, "bronze_to_silver", load_type, batch_id,
                f"{tables['bronze']} [batch_id={batch_id}]",
                lambda: load_bronze_to_silver(tables, batch_id)
            ))

    # Gold is refreshed at the end of every run that loaded Silver, so a new incremental
    # file goes all the way to the dashboard tables in one run.
    silver_loaded = bool(results) and results[-1]["layer"] == "bronze_to_silver" and results[-1]["status"] == "SUCCESS"
    if layer == "gold" or silver_loaded:
        results.extend(refresh_gold(tables, run_id, load_type, batch_id, data_dir))

    failed = [r for r in results if r["status"] != "SUCCESS"]
    if failed:
        raise RuntimeError(
            f"{len(failed)} of {len(results)} step(s) failed in run {run_id}. "
            f"See {tables['logs']} for details."
        )
    return run_id, tables